# YOLO 비교 — yolov8s / yolov11n / yolov11s

런타임 → 런타임 유형 변경 → T4 GPU

`yolov8n`은 `train_yolov8n_colab.ipynb`에서 따로. `QUICK_MODE` 값을 양쪽 동일하게 둘 것.

In [ ]:
%pip install -q ultralytics

import torch
assert torch.cuda.is_available(), "GPU 런타임 아님"
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

RUNS_DIR = Path('/content/runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(RUNS_DIR)

In [ ]:
from google.colab import files

DATASET_ZIP = 'Pipe Crack Detection.v8-pipe-crack.yolov8.zip'

ZIP_PATH = next((p for p in (Path.cwd() / DATASET_ZIP, Path('/content') / DATASET_ZIP)
                 if p.is_file()), None)

if ZIP_PATH is None:
    print(f"{DATASET_ZIP} 선택")
    uploaded = files.upload()
    ZIP_PATH = Path('/content') / next(iter(uploaded))

print(ZIP_PATH, f"{ZIP_PATH.stat().st_size / 1e6:.1f} MB")

In [ ]:
import shutil, yaml

DATASET_DIR = Path('/content/dataset')
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), str(DATASET_DIR))

DATA_YAML = DATASET_DIR / 'data.yaml'
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg.pop('path', None)
for key, split in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
    images = DATASET_DIR / split / 'images'
    if images.is_dir():
        cfg[key] = str(images)
    else:
        cfg.pop(key, None)
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

CLASS_NAMES = cfg['names']
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES)]
CRACK_IDX = CLASS_NAMES.index('Crack') if 'Crack' in CLASS_NAMES else 0

for split in ('train', 'valid', 'test'):
    print(split, len(list((DATASET_DIR / split / 'images').glob('*'))))
print(CLASS_NAMES, '| Crack idx', CRACK_IDX)

In [ ]:
MODELS = ['yolov8s', 'yolov11n', 'yolov11s']

WEIGHT_ALIAS = {
    'yolov8n': 'yolov8n.pt',
    'yolov8s': 'yolov8s.pt',
    'yolov11n': 'yolo11n.pt',
    'yolov11s': 'yolo11s.pt',
}

QUICK_MODE = True

TRAIN_ARGS = dict(
    data=str(DATA_YAML),
    epochs=25 if QUICK_MODE else 60,
    patience=10 if QUICK_MODE else 20,
    imgsz=640,
    batch=16,
    seed=0,
    mosaic=0.5,
    close_mosaic=10,
    project=str(RUNS_DIR),
    exist_ok=True,
    plots=True,
    verbose=False,
)

TARGET_HZ = 10.0
LATENCY_BUDGET_MS = 1000.0 / TARGET_HZ

print(MODELS, '| epochs', TRAIN_ARGS['epochs'], '| budget', LATENCY_BUDGET_MS, 'ms')

In [ ]:
import time, gc, json, traceback
from ultralytics import YOLO

PROGRESS = RUNS_DIR / 'progress.json'
runs = json.loads(PROGRESS.read_text()) if PROGRESS.exists() else {}

for tag in MODELS:
    if tag in runs and (RUNS_DIR / tag / 'weights' / 'best.pt').exists():
        print(f"skip {tag}")
        continue

    print(f"\n===== {tag} ({WEIGHT_ALIAS[tag]}) =====")
    model = None
    try:
        started = time.time()
        model = YOLO(WEIGHT_ALIAS[tag])
        result = model.train(name=tag, **TRAIN_ARGS)
        elapsed = time.time() - started
        runs[tag] = {'save_dir': str(result.save_dir), 'train_sec': elapsed}
        PROGRESS.write_text(json.dumps(runs, indent=2))
        print(f"{tag} {elapsed / 60:.1f}분")
    except Exception:
        traceback.print_exc()
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

print(list(runs))

In [ ]:
import numpy as np
import pandas as pd

test_images = [str(p) for p in sorted((DATASET_DIR / 'test' / 'images').glob('*.jpg'))]
LATENCY_SAMPLES = test_images[:60]
trained = [t for t in MODELS if t in runs]
rows = []

for tag in trained:
    model = YOLO(str(Path(runs[tag]['save_dir']) / 'weights' / 'best.pt'))

    try:
        _, n_params, _, gflops = model.info(verbose=False)
    except Exception:
        n_params, gflops = float('nan'), float('nan')

    m = model.val(data=str(DATA_YAML), split='test', imgsz=640, verbose=False)
    try:
        pos = list(m.box.ap_class_index).index(CRACK_IDX)
        crack_map50, crack_map = float(m.box.ap50[pos]), float(m.box.ap[pos])
    except (ValueError, IndexError):
        crack_map50 = crack_map = float('nan')

    model.predict(source=LATENCY_SAMPLES, imgsz=640, verbose=False)
    preds = model.predict(source=LATENCY_SAMPLES, imgsz=640, verbose=False)
    s = preds[0].speed
    latency = s['preprocess'] + s['inference'] + s['postprocess']

    cc = [p.boxes.conf[p.boxes.cls == CRACK_IDX].cpu().numpy() for p in preds]
    cc = np.concatenate([c for c in cc if len(c)]) if any(len(c) for c in cc) else np.array([])

    rows.append({
        '모델': tag,
        'Crack mAP50': round(crack_map50, 4),
        'Crack mAP50-95': round(crack_map, 4),
        '전체 mAP50': round(float(m.box.map50), 4),
        '전체 mAP50-95': round(float(m.box.map), 4),
        'P': round(float(m.box.mp), 4),
        'R': round(float(m.box.mr), 4),
        '지연(ms)': round(latency, 1),
        '파라미터(M)': round(n_params / 1e6, 2),
        'GFLOPs': round(gflops, 1),
        '학습(분)': round(runs[tag]['train_sec'] / 60, 1),
        'Crack conf>=0.8': int((cc >= 0.8).sum()),
        'Crack 검출수': int(len(cc)),
        'epochs': TRAIN_ARGS['epochs'],
    })

    del model
    gc.collect()
    torch.cuda.empty_cache()

df = pd.DataFrame(rows).sort_values('Crack mAP50-95', ascending=False).reset_index(drop=True)
df['10Hz'] = df['지연(ms)'] <= LATENCY_BUDGET_MS
df.to_csv(RUNS_DIR / 'comparison.csv', index=False)
df

In [ ]:
passed = df[df['10Hz']]
winner = passed.iloc[0]['모델'] if len(passed) else df.iloc[0]['모델']
print('선정:', winner, '' if len(passed) else '(지연 예산 초과)')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
x = range(len(df))

axes[0].bar(x, df['Crack mAP50-95'], color='steelblue')
axes[0].bar(x, df['Crack mAP50'], color='steelblue', alpha=0.35)
axes[0].set_title('Crack mAP50-95 / mAP50')

axes[1].bar(x, df['지연(ms)'], color=['seagreen' if ok else 'indianred' for ok in df['10Hz']])
axes[1].axhline(LATENCY_BUDGET_MS, color='black', linestyle='--', linewidth=1)
axes[1].set_title('latency (ms)')

axes[2].scatter(df['파라미터(M)'], df['Crack mAP50-95'], s=110, color='darkorange')
for _, r in df.iterrows():
    axes[2].annotate(r['모델'], (r['파라미터(M)'], r['Crack mAP50-95']),
                     textcoords='offset points', xytext=(6, 5), fontsize=9)
axes[2].set_title('params vs mAP')
axes[2].set_xlabel('params (M)')

for ax in axes[:2]:
    ax.set_xticks(list(x))
    ax.set_xticklabels(df['모델'], rotation=15)

plt.tight_layout()
plt.savefig(RUNS_DIR / 'comparison.png', dpi=130)
plt.show()

In [ ]:
SAMPLES = 3
fig, axes = plt.subplots(len(trained), SAMPLES, figsize=(4.5 * SAMPLES, 4.5 * len(trained)))
axes = np.atleast_2d(axes)

for row, tag in enumerate(trained):
    model = YOLO(str(Path(runs[tag]['save_dir']) / 'weights' / 'best.pt'))
    preds = model.predict(source=test_images[:SAMPLES], imgsz=640, conf=0.25, verbose=False)
    for col, pred in enumerate(preds):
        axes[row, col].imshow(pred.plot()[:, :, ::-1])
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(tag, loc='left', fontweight='bold')
    del model
    gc.collect()
    torch.cuda.empty_cache()

plt.tight_layout()
plt.show()

In [ ]:
OUT = Path('/content/model_comparison')
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()

df.to_csv(OUT / 'comparison.csv', index=False)
if (RUNS_DIR / 'comparison.png').exists():
    shutil.copy(RUNS_DIR / 'comparison.png', OUT / 'comparison.png')

for tag in trained:
    d = Path(runs[tag]['save_dir'])
    shutil.copy(d / 'weights' / 'best.pt', OUT / f'{tag}_best.pt')
    for f in ('results.png', 'results.csv', 'confusion_matrix_normalized.png', 'PR_curve.png'):
        if (d / f).exists():
            shutil.copy(d / f, OUT / f'{tag}_{f}')

archive = shutil.make_archive('/content/model_comparison', 'zip', str(OUT))
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
print(sorted(p.name for p in OUT.iterdir()))

files.download(archive)

In [ ]:
model = YOLO(str(Path(runs[winner]['save_dir']) / 'weights' / 'best.pt'))
onnx_path = model.export(format='onnx', imgsz=640, simplify=True)
files.download(str(onnx_path))

## 8n 결과와 합치기

```python
import pandas as pd
pd.concat([pd.read_csv('comparison.csv'), pd.read_csv('yolov8n_result.csv')]) \
  .sort_values('Crack mAP50-95', ascending=False)
```